In [87]:
import random
import math
from collections import defaultdict
import pandas as pd

# Problem data
customers = [(1, 1), (2, 4), (3, 2), (6, 6)]
facilities = [(1, 3), (3, 3), (5, 1), (7, 5), (6, 2), (2, 5), (0, 0), (4, 2), (5, 4), (6, 1)]
p = 4
Degprt = 0.33
NR = NA = max(1, int(p * Degprt))
OUTPUT_FILE = 'heuristic_training_data.csv'
NUM_SAMPLES = 100  #training samples sizeee

def distance(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)


def LH1(solution):
    s = list(solution)
    # Random removal and greedy add
    open_indices = [i for i, bit in enumerate(s) if bit == '1']
    removed = random.sample(open_indices, min(NR, len(open_indices)))
    for idx in removed:
        s[idx] = '0'
    
    # Greedy addition
    for _ in range(NR):
        closed_indices = [i for i, bit in enumerate(s) if bit == '0']
        best_idx = None
        best_improvement = -float('inf')
        for idx in closed_indices:
            s[idx] = '1'
            current_cost = evaluate_solution(s)
            s[idx] = '0'
            
            improvement = evaluate_solution(s) - current_cost
            if improvement > best_improvement:
                best_improvement = improvement
                best_idx = idx
        
        if best_idx is not None:
            s[best_idx] = '1'
    
    return ''.join(s)

def LH2(solution):
    s = list(solution)
    # Random addition and greedy removal
    closed_indices = [i for i, bit in enumerate(s) if bit == '0']
    added = random.sample(closed_indices, min(NA, len(closed_indices)))
    for idx in added:
        s[idx] = '1'
    
    # Greedy removal
    for _ in range(NA):
        adj = compute_adjacency(s)
        open_indices = [i for i, bit in enumerate(s) if bit == '1']
        if not open_indices:
            break
        worst_idx = min(open_indices, key=lambda idx: adj.get(idx, 0))
        s[worst_idx] = '0'
    
    return ''.join(s)

def LH3(solution):
    s = list(solution)
    # Greedy removal and greedy add
    for _ in range(NR):
        adj = compute_adjacency(s)
        open_indices = [i for i, bit in enumerate(s) if bit == '1']
        if not open_indices:
            break
        worst_idx = min(open_indices, key=lambda idx: adj.get(idx, 0))
        s[worst_idx] = '0'
    # Greedy addition
    for _ in range(NR):
        closed_indices = [i for i, bit in enumerate(s) if bit == '0']
        best_idx = None
        best_improvement = -float('inf')
        
        for idx in closed_indices:
            s[idx] = '1'
            current_cost = evaluate_solution(s)
            s[idx] = '0'
            
            improvement = evaluate_solution(s) - current_cost
            if improvement > best_improvement:
                best_improvement = improvement
                best_idx = idx
        
        if best_idx is not None:
            s[best_idx] = '1'
    return ''.join(s)

def LH4(solution):
    s = list(solution)
    # Greedy addition and greedy removal
    for _ in range(NA):
        closed_indices = [i for i, bit in enumerate(s) if bit == '0']
        best_idx = None
        best_improvement = -float('inf')
        
        for idx in closed_indices:
            s[idx] = '1'
            current_cost = evaluate_solution(s)
            s[idx] = '0'
            
            improvement = evaluate_solution(s) - current_cost
            if improvement > best_improvement:
                best_improvement = improvement
                best_idx = idx
        
        if best_idx is not None:
            s[best_idx] = '1'
    
    # Greedy removal
    for _ in range(NA):
        adj = compute_adjacency(s)
        open_indices = [i for i, bit in enumerate(s) if bit == '1']
        if not open_indices:
            break
        worst_idx = min(open_indices, key=lambda idx: adj.get(idx, 0))
        s[worst_idx] = '0'
    
    return ''.join(s)


def evaluate_solution(solution):
    open_facilities = []
    for i in range(len(solution)):
        bit = solution[i]
        if bit == '1':
            open_facilities.append(facilities[i])
    total_cost = 0
    for cust in customers:
        min_dist = float('inf')
        for fac in open_facilities:
            d = distance(cust, fac)
            if d < min_dist:
                min_dist = d
        total_cost += min_dist

    return total_cost

def compute_adjacency(solution):
    counts = defaultdict(int)
    open_indices = []
    for i in range(len(solution)):
        if solution[i] == '1':
            open_indices.append(i)

    for cust in customers:
        min_dist = float('inf')
        closest = None
        for idx in open_indices:
            d = distance(cust, facilities[idx])
            if d < min_dist:
                min_dist = d
                closest = idx
        counts[closest] += 1

    return counts

def generate_initial_solution():
    solution = []
    for _ in range(len(facilities)):
        solution.append('0')
    sample_indices = random.sample(range(len(facilities)), p)
    for i in sample_indices:
        solution[i] = '1'
    return ''.join(solution)

def find_best_heuristic(solution):
    original_cost = evaluate_solution(solution)
    best_heuristic = None
    best_cost = float('inf')

    heuristics = [('LH1', LH1), ('LH2', LH2), ('LH3', LH3), ('LH4', LH4)]
    for i in range(len(heuristics)):
        name = heuristics[i][0]
        heuristic = heuristics[i][1]
        current_cost = evaluate_solution(heuristic(solution))
        if current_cost < best_cost:
            best_cost = current_cost
            best_heuristic = name

    return best_heuristic

def generate_training_data():
    data = []
    for _ in range(NUM_SAMPLES):
        solution = generate_initial_solution()
        best_heuristic = find_best_heuristic(solution)
        row = {}
        row['solution'] = solution
        row['best_heuristic'] = best_heuristic
        data.append(row)
    return pd.DataFrame(data)

if __name__ == "__main__":
    df = generate_training_data()
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved to {OUTPUT_FILE}")

Saved to heuristic_training_data.csv


In [88]:
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
import pickle

df = pd.read_csv('heuristic_training_data.csv')
df['solution'] = df['solution'].astype(str)
X = df['solution']
y = df['best_heuristic']

vectorizer = CountVectorizer(analyzer='char', binary=True, lowercase=False)
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

model = MultinomialNB()
model.fit(X_train, y_train)

with open('heuristic_selector.pkl', 'wb') as f:
    pickle.dump({
        'model': model,
        'vectorizer': vectorizer,
        'classes': model.classes_
    }, f)

def predict_best_heuristic(solution_string):
    vec = vectorizer.transform([str(solution_string)])
    return model.predict(vec)[0]

test_solution = '1010010001'
print(f"\nPredicted best heuristic for {test_solution}: {predict_best_heuristic(test_solution)}")
print(f"Available classes: {model.classes_}")



Predicted best heuristic for 1010010001: LH3
Available classes: ['LH1' 'LH2' 'LH3' 'LH4']


In [89]:
import pandas as pd
import pickle

with open('heuristic_selector.pkl', 'rb') as f:
    saved = pickle.load(f)
    model = saved['model']
    vectorizer = saved['vectorizer']

heuristics = {
    'LH1': LH1,
    'LH2': LH2, 
    'LH3': LH3,
    'LH4': LH4
}

def optimize_solution(initial_solution, max_iterations=100):
    history = []
    current_solution = initial_solution
    current_cost = evaluate_solution(current_solution)
    no_improvement_count = 0

    for iteration in range(max_iterations):
        heuristic_name = model.predict(vectorizer.transform([current_solution]))[0]
        new_solution = heuristics[heuristic_name](current_solution)
        new_cost = evaluate_solution(new_solution)

        history.append({
            'iteration': iteration,
            'solution': current_solution,
            'heuristic': heuristic_name,
            'new_solution': new_solution,
            'cost': current_cost,
            'new_cost': new_cost,
            'improvement': current_cost - new_cost
        })

        if new_cost >= current_cost:
            no_improvement_count += 1
            if no_improvement_count >= 3:
                break
        else:
            no_improvement_count = 0

        current_solution = new_solution
        current_cost = new_cost

    return pd.DataFrame(history)

if __name__ == "__main__":
    initial_solution = generate_initial_solution()
    results_df = optimize_solution(initial_solution)
    results_df.to_csv('optimization_path.csv', index=False)


In [90]:
import random
import pandas as pd



def get_top2_heuristics(solution):
    #Returns names of top 2 heuristics
    original_cost = evaluate_solution(solution)
    results = []
    
    for name, heuristic in [('LH1', LH1), ('LH2', LH2), ('LH3', LH3), ('LH4', LH4)]:
        new_cost = evaluate_solution(heuristic(solution))
        results.append((name, original_cost - new_cost))
    results.sort(key=lambda x: x[1], reverse=True)
    return [results[0][0], results[1][0]] 

def generate_training_data():
    data = []
    for _ in range(1000):  # Change sam,ple size
        solution = generate_initial_solution()
        best, second_best = get_top2_heuristics(solution)
        data.append({
            'solution': solution,
            'best_heuristic': best,
            'second_best_heuristic': second_best
        })
    return pd.DataFrame(data)

df = generate_training_data()
df.to_csv('top2_heuristics_clean.csv', index=False)
print(f"Saved {len(df)} solutions with top 2 heuristics")
print(df.head())

Saved 1000 solutions with top 2 heuristics
     solution best_heuristic second_best_heuristic
0  0111010000            LH3                   LH4
1  1100010100            LH3                   LH4
2  0010000111            LH2                   LH3
3  0100110001            LH1                   LH2
4  0100100011            LH3                   LH4


In [91]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer
df = pd.read_csv('top2_heuristics_clean.csv')
df['solution'] = df['solution'].astype(str).str.strip()
y = df.apply(lambda x: [x['best_heuristic'], x['second_best_heuristic']], axis=1).tolist()
vectorizer = CountVectorizer(
    analyzer='char',
    binary=True,
    lowercase=False,
    token_pattern='[01]'
)
X = vectorizer.fit_transform(df['solution'])

# Multi-label setup
mlb = MultiLabelBinarizer(classes=['LH1', 'LH2', 'LH3', 'LH4'])
y_encoded = mlb.fit_transform(y)

#Train model
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X, y_encoded)

#prediction function
def predict_top2(solution):
    solution = str(solution).strip()
    if not all(c in '01' for c in solution):
        return ('LH1', 'LH2') 
        
    vec = vectorizer.transform([solution])
    pred = dt.predict(vec)
    heuristics = mlb.inverse_transform(pred)[0]
    
    # exactly 2 heuristics
    if len(heuristics) == 0:
        return ('LH1', 'LH2')
    elif len(heuristics) == 1:
        return (heuristics[0], 'LH1')
    return tuple(heuristics[:2])

# Generate and save predictions
results = []
for sol in df['solution'].unique():
    try:
        h1, h2 = predict_top2(sol)
        results.append({
            'solution': sol,
            'best_heuristic': h1,
            'second_heuristic': h2
        })
    except Exception as e:
        print(f"Error with {sol[:50]}...: {e}")

pd.DataFrame(results).to_csv('reliable_top2_predictions.csv', index=False)
print(f"Saved {len(results)} predictions")

Saved 209 predictions


c:\Users\Dvns\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:547: UserWarning: The parameter 'token_pattern' will not be used since 'analyzer' != 'word'
  warnings.warn(


In [ ]:
import pandas as pd
predictions = pd.read_csv('reliable_top2_predictions.csv')
predictions['solution'] = predictions['solution'].astype(str).str.strip()

def apply_best_heuristics(solution):
    """Apply both predicted heuristics with exact string matching"""
    sol_str = str(solution).strip()
    
    # Exact match lookup
    pred = predictions[predictions['solution'] == sol_str]
    
    if len(pred) == 0:
        print(f"\n No prediction found for: {sol_str}")
        print("Available solutions start with:")
        print(predictions['solution'].head(3).to_string(index=False))
        return solution #ORIGINAL WHEN NO MATCH
    
    h1 = pred.iloc[0]['best_heuristic']
    h2 = pred.iloc[0]['second_heuristic']
    
    print(f"\n🔹 Original: {sol_str} (Cost: {evaluate_solution(sol_str):.2f})")
    print(f"Predicted heuristics: {h1} → {h2}")
    
    try:
        # Apply first heuristic
        step1 = globals()[h1](sol_str)
        print(f"After {h1}: {step1} (Cost: {evaluate_solution(step1):.2f})")
        # Apply second heuristic
        step2 = globals()[h2](step1)
        print(f"After {h2}: {step2} (Cost: {evaluate_solution(step2):.2f})")
        improvement = evaluate_solution(sol_str) - evaluate_solution(step2)
        print(f"✅ Total improvement: {improvement:.2f}")
        return step2
    
    except Exception as e:
        print(f"Error applying heuristics: {e}")
        return sol_str

# Example test
test_solutions = [
    '1010010001', 
    '1111000000',  # Test case
    predictions.iloc[0]['solution'] 
]

for sol in test_solutions:
    optimized = apply_best_heuristics(sol)
    print(f"Final optimized: {optimized}\n{'-'*40}")


🔹 Original: 1010010001 (Cost: 9.36)
Predicted heuristics: LH3 → LH1
After LH3: 1001010001 (Cost: 6.65)
After LH1: 1101000001 (Cost: 5.83)
✅ Total improvement: 3.53
Final optimized: 1101000001
----------------------------------------

🔹 Original: 1111000000 (Cost: 5.83)
Predicted heuristics: LH3 → LH1
After LH3: 1101001000 (Cost: 5.24)
After LH1: 1101001000 (Cost: 5.24)
✅ Total improvement: 0.59
Final optimized: 1101001000
----------------------------------------

🔹 Original: 111010000 (Cost: 8.41)
Predicted heuristics: LH3 → LH1
After LH3: 110110000 (Cost: 5.83)
After LH1: 010110100 (Cost: 5.24)
✅ Total improvement: 3.17
Final optimized: 010110100
----------------------------------------
